In [2]:
!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install matplotlib pandas numpy scikit-learn yfinance pandas-ta tqdm seaborn plotly ipywidgets

Looking in indexes: https://download.pytorch.org/whl/cu118


In [54]:
# Core imports
import os
import random
import warnings

import numpy as np

warnings.filterwarnings('ignore')

# Data visualization
%matplotlib inline

# Machine learning
import torch
import pandas as pd

In [4]:
TINKOFF_API_PROD = 'invest-public-api.tinkoff.ru:443'
TINKOFF_API_SANDBOX = 'sandbox-invest-public-api.tinkoff.ru:443'

In [6]:
INVEST_API_KEY = input("Enter ML developer Invest API Key: ")

In [7]:
# Check CUDA availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")
print(f"Current CUDA device: {torch.cuda.current_device()}")
print(f"CUDA device name: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.7.0+cu118
CUDA available: True
CUDA device count: 1
Current CUDA device: 0
CUDA device name: NVIDIA GeForce RTX 4080 SUPER


In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [9]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed()

In [90]:
from TInvestDataProvider import TInvestDataProvider
from datetime import datetime, timedelta
from externalClients.TInvestApi.proto.marketdata_pb2 import (
    CandleInterval
)



In [91]:
import pytz
from datetime import datetime

moscow_tz = pytz.timezone('Europe/Moscow')
naive_datetime = datetime(year=2025, month=4, day=24, hour=11, second=1)
moscow_datetime = moscow_tz.localize(naive_datetime)
train_period_end = moscow_datetime.astimezone(pytz.UTC)

In [77]:
provider = TInvestDataProvider(api_key=INVEST_API_KEY)
TARGET_CANDLE_COUNT = 80_000
timestep = timedelta(hours=40)

df = None

try:
    current_end = train_period_end
    step_count = 1
    while True:
        print(f"Step {step_count}. ", end="")
        step_count += 1
        current_start = current_end - timestep

        new_df = await provider.get_historical_candles(
            instrument_id="e6123145-9665-43e0-8413-cd61b8aa9b13",
            from_time=current_start,
            to_time=current_end,
            interval=CandleInterval.CANDLE_INTERVAL_1_MIN
        )

        if df is None:
            df = new_df
        else:
            df = pd.concat([new_df, df])

        if len(df) >= TARGET_CANDLE_COUNT:
            df = df.iloc[-TARGET_CANDLE_COUNT:]
            print(f"Current df shape: {df.shape}")
            break

        current_end = current_start
        print(f"Current df shape: {df.shape}")

finally:
    await provider.close()


Step 1. Current df shape: (1520, 7)
Step 2. Current df shape: (3218, 7)
Step 3. Current df shape: (4966, 7)
Step 4. Current df shape: (6545, 7)
Step 5. Current df shape: (8243, 7)
Step 6. Current df shape: (9986, 7)
Step 7. Current df shape: (11535, 7)
Step 8. Current df shape: (13395, 7)
Step 9. Current df shape: (15138, 7)
Step 10. Current df shape: (16658, 7)
Step 11. Current df shape: (18473, 7)
Step 12. Current df shape: (20485, 7)
Step 13. Current df shape: (22005, 7)
Step 14. Current df shape: (23703, 7)
Step 15. Current df shape: (25459, 7)
Step 16. Current df shape: (27219, 7)
Step 17. Current df shape: (28917, 7)
Step 18. Current df shape: (30659, 7)
Step 19. Current df shape: (32120, 7)
Step 20. Current df shape: (33799, 7)
Step 21. Current df shape: (35568, 7)
Step 22. Current df shape: (37088, 7)
Step 23. Current df shape: (38786, 7)
Step 24. Current df shape: (40553, 7)
Step 25. Current df shape: (42189, 7)
Step 26. Current df shape: (43886, 7)
Step 27. Current df shape: 

In [103]:
df.shape

(80000, 7)

In [105]:
from FeatureEngineer import FeatureEngineer

feature_engineer = FeatureEngineer(lookback_window=60)
augmented_df = feature_engineer.add_features(df)

In [110]:
!pip install numpy --upgrade

  Using cached numpy-2.2.5-cp311-cp311-win_amd64.whl.metadata (60 kB)
Using cached numpy-2.2.5-cp311-cp311-win_amd64.whl (12.9 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.0 requires numpy<2.2,>=1.24, but you have numpy 2.2.5 which is incompatible.
pandas-ta 0.3.14b0 requires numpy==1.26.4, but you have numpy 2.2.5 which is incompatible.


In [117]:
!pip install -r requirements.txt  --exists-action w

   ---------------------------------------- 0.0/14.6 MB ? eta -:--:--
   ----- ---------------------------------- 1.8/14.6 MB 14.4 MB/s eta 0:00:01
   ------------------------------ --------- 11.0/14.6 MB 34.4 MB/s eta 0:00:01
   ---------------------------------------- 14.6/14.6 MB 36.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/42.2 MB ? eta -:--:--
   ----- ---------------------------------- 6.3/42.2 MB 32.1 MB/s eta 0:00:02
   ------------- -------------------------- 14.7/42.2 MB 35.5 MB/s eta 0:00:01
   --------------------- ------------------ 22.3/42.2 MB 38.1 MB/s eta 0:00:01
   ------------------------------ --------- 31.7/42.2 MB 38.0 MB/s eta 0:00:01
   ---------------------------------------  41.7/42.2 MB 40.8 MB/s eta 0:00:01
   ---------------------------------------- 42.2/42.2 MB 38.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 8.3/8.3 MB 42.8 MB/s eta 0:00:00
   --

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.0 requires numpy<2.2,>=1.24, but you have numpy 1.23.5 which is incompatible.
pandas-ta 0.3.14b0 requires numpy==1.26.4, but you have numpy 1.23.5 which is incompatible.


In [123]:
!pip uninstall numpy scipy scikit-learn torch torchvision torchaudio -y

Found existing installation: numpy 1.23.5
Uninstalling numpy-1.23.5:
  Successfully uninstalled numpy-1.23.5
Found existing installation: scipy 1.10.1
Uninstalling scipy-1.10.1:
  Successfully uninstalled scipy-1.10.1
Found existing installation: scikit-learn 1.2.2
Uninstalling scikit-learn-1.2.2:
  Successfully uninstalled scikit-learn-1.2.2
Found existing installation: torch 2.0.1
Uninstalling torch-2.0.1:
  Successfully uninstalled torch-2.0.1
Found existing installation: torchvision 0.15.2
Uninstalling torchvision-0.15.2:
  Successfully uninstalled torchvision-0.15.2
Found existing installation: torchaudio 2.0.2
Uninstalling torchaudio-2.0.2:
  Successfully uninstalled torchaudio-2.0.2


You can safely remove it manually.


In [141]:
!pip install numpy==1.26.4 scipy==1.14.1 scikit-learn==1.2.2
!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install pandas==1.3.5 tqdm==4.64.1

  Using cached scipy-1.14.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
   ---------------------------------------- 0.0/44.8 MB ? eta -:--:--
    --------------------------------------- 0.8/44.8 MB 8.5 MB/s eta 0:00:06
   ------- -------------------------------- 8.9/44.8 MB 30.8 MB/s eta 0:00:02
   --------------- ------------------------ 17.8/44.8 MB 38.8 MB/s eta 0:00:01
   ------------------------- -------------- 28.0/44.8 MB 41.4 MB/s eta 0:00:01
   ---------------------------------- ----- 38.5/44.8 MB 43.7 MB/s eta 0:00:01
   ---------------------------------------- 44.8/44.8 MB 42.5 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.15.2
    Uninstalling scipy-1.15.2:
      Successfully uninstalled scipy-1.15.2
Looking in indexes: https://download.pytorch.org/whl/cu118


In [143]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from tqdm import tqdm
import numpy as np

# Configuration
class Config:
    SEED = 42
    BATCH_SIZE = 256
    HIDDEN_SIZE = 128
    NUM_LAYERS = 2
    DROPOUT = 0.3
    LEARNING_RATE = 1e-3
    EPOCHS = 100
    PATIENCE = 5
    SEQ_LEN = 60  # Lookback window size


In [144]:
class LSTMAttention(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=Config.DROPOUT if num_layers > 1 else 0
        )

        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1),
            nn.Softmax(dim=1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size//2),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(hidden_size//2, output_size)
        )

    def forward(self, x):
        # x shape: (batch_size, seq_len, input_size)
        lstm_out, _ = self.lstm(x)  # (batch_size, seq_len, hidden_size)

        # Attention weights
        attn_weights = self.attention(lstm_out)  # (batch_size, seq_len, 1)
        context = torch.sum(attn_weights * lstm_out, dim=1)  # (batch_size, hidden_size)

        return self.classifier(context)



In [145]:
# Training Utilities
class EarlyStopper:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = float('inf')

    def __call__(self, validation_loss):
        if validation_loss < self.min_validation_loss - self.min_delta:
            self.min_validation_loss = validation_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False

def create_sequences(data, seq_length):
    sequences = []
    targets = []
    for i in range(len(data)-seq_length):
        sequences.append(data[i:i+seq_length, :-1])  # Exclude target
        targets.append(data[i+seq_length, -1])       # Last element is target
    return np.array(sequences), np.array(targets)


In [147]:
def train_test_split(X, y, test_size=0.2, shuffle=True, random_state=None):
    """Custom train-test split implementation without scikit-learn"""
    if random_state:
        np.random.seed(random_state)

    n_samples = len(X)
    test_samples = int(n_samples * test_size)

    if shuffle:
        indices = np.random.permutation(n_samples)
    else:
        indices = np.arange(n_samples)

    test_indices = indices[:test_samples]
    train_indices = indices[test_samples:]

    X_train, X_test = X[train_indices], X[test_indices]
    y_train, y_test = y[train_indices], y[test_indices]

    return X_train, X_test, y_train, y_test

In [148]:
# Training Loop
def train_model(df):
    # Prepare data
    data = df.values.astype(np.float32)
    X, y = create_sequences(data, Config.SEQ_LEN)

    # Split data
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=Config.SEED, shuffle=False)

    # Convert to tensors
    train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    val_dataset = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))

    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE)

    # Initialize model
    input_size = X_train.shape[-1]
    model = LSTMAttention(
        input_size=input_size,
        hidden_size=Config.HIDDEN_SIZE,
        num_layers=Config.NUM_LAYERS,
        output_size=2  # Buy/Sell (binary classification)
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2)
    early_stopper = EarlyStopper(patience=Config.PATIENCE)

    best_val_loss = float('inf')
    try:
        for epoch in range(Config.EPOCHS):
            # Training
            model.train()
            train_loss = 0
            progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{Config.EPOCHS} [Train]')
            for inputs, targets in progress_bar:
                inputs, targets = inputs.to(device), targets.long().to(device)

                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

                train_loss += loss.item()
                progress_bar.set_postfix(loss=loss.item())

            # Validation
            model.eval()
            val_loss = 0
            correct = 0
            total = 0
            with torch.no_grad():
                val_progress = tqdm(val_loader, desc=f'Epoch {epoch+1}/{Config.EPOCHS} [Val]')
                for inputs, targets in val_progress:
                    inputs, targets = inputs.to(device), targets.long().to(device)

                    outputs = model(inputs)
                    loss = criterion(outputs, targets)
                    val_loss += loss.item()

                    _, predicted = torch.max(outputs.data, 1)
                    total += targets.size(0)
                    correct += (predicted == targets).sum().item()

                    val_progress.set_postfix(loss=loss.item(), acc=100*correct/total)

            # Epoch statistics
            train_loss /= len(train_loader)
            val_loss /= len(val_loader)
            val_acc = 100 * correct / total

            print(f'Epoch {epoch+1} | '
                  f'Train Loss: {train_loss:.4f} | '
                  f'Val Loss: {val_loss:.4f} | '
                  f'Val Acc: {val_acc:.2f}%')

            # Early stopping and model checkpoint
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(model.state_dict(), 'best_model.pth')
                print(f'New best model saved with val loss {val_loss:.4f}')

            if early_stopper(val_loss):
                print(f'Early stopping at epoch {epoch+1}')
                break

            scheduler.step(val_loss)

    except KeyboardInterrupt:
        print("\nTraining interrupted by user. Saving current model...")
        torch.save(model.state_dict(), 'interrupted_model.pth')

    return model


In [151]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

In [154]:
df.columns

Index(['open', 'high', 'low', 'close', 'volume', 'is_complete', 'source'], dtype='object')

In [ ]:
print("Unique target values:", df['target_direction'].unique())
print("Target value counts:\n", df['target_direction'].value_counts())

In [155]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Assuming df is your preprocessed DataFrame
model = train_model(augmented_df)

# Test inference
sample_input = torch.randn(1, Config.SEQ_LEN, 25).to(device)  # 25 features
with torch.no_grad():
    output = model(sample_input)
print("Sample output:", torch.softmax(output, dim=1).cpu().numpy())

Using device: cuda


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
